In [1]:
import os, sys

# Repository information
REPO_NAME = "RecSys-Challenge-2025"
REPO_URL  = f"github.com/Lv1g1/{REPO_NAME}.git"

# Detect environment
IS_COLAB = 'content' in os.getcwd()
IS_KAGGLE = 'kaggle' in os.getcwd()
IS_LOCAL = not (IS_COLAB or IS_KAGGLE)

if IS_COLAB:
    WORKING_DIR = "/content"

    # Mount Google Drive
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)

    # Get GitHub token via input
    def get_token():
        from getpass import getpass
        return getpass("GitHub Token: ")

elif IS_KAGGLE:
    WORKING_DIR = "/kaggle/working"

    # Get GitHub token from Kaggle secrets
    def get_token():
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret("Token")

# If local environment assume inside the repo
LOCAL_REPO_PATH = os.getcwd() if IS_LOCAL else os.path.join(WORKING_DIR, REPO_NAME)

# Clone the repository if it doesn't exist
if not os.path.exists(LOCAL_REPO_PATH):
    os.chdir(WORKING_DIR)
    token = get_token()

    !git clone https://{token}@{REPO_URL}
else:
    print("Repo already exists — pulling latest changes")
    os.chdir(LOCAL_REPO_PATH)
    !git pull
    os.chdir(WORKING_DIR)

# Add to Python PATH
if LOCAL_REPO_PATH not in sys.path:
    sys.path.append(LOCAL_REPO_PATH)

Repo already exists — pulling latest changes
Already up to date.


In [2]:
%%capture

# Compile Cython files
os.chdir(LOCAL_REPO_PATH)
!python Challenge/compile_cython.py . --inplace
os.chdir(WORKING_DIR)

In [3]:
%%capture

!pip install optuna
import optuna

In [4]:
import importlib
import scipy.sparse as sps
import pandas as pd
import numpy as np

from Challenge import paths
importlib.reload(paths)

from Challenge.hyper_tuning import hyperparameter_tuning

Running on kaggle — storage at: /kaggle/working
Running on kaggle — storage at: /kaggle/working


In [5]:
# Load datasets
URM_train = sps.load_npz(paths.URM_TRAIN)
URM_validation = sps.load_npz(paths.URM_VALIDATION)

In [6]:
def evaluate_recommender(recommender, at):
    cumulative_recall = 0.0
    num_eval = 0
    
    for user_id in range(URM_validation.shape[0]):
        relevant_items = URM_validation.indices[URM_validation.indptr[user_id]:URM_validation.indptr[user_id+1]]
        
        if len(relevant_items)>0:
            num_eval+=1
            
            recommended_items = recommender.recommend(user_id, cutoff=at)
            
            is_relevant = np.isin(recommended_items, relevant_items, assume_unique=True)
            recall_score = np.sum(is_relevant, dtype=np.float32) / relevant_items.shape[0]

            cumulative_recall += recall_score

    return cumulative_recall / num_eval

In [7]:
from Recommenders.MatrixFactorization.IALSRecommender import IALSRecommender

# Load IALS model
ials_model = IALSRecommender(URM_train)
ials_model.load_model(paths.MODEL_DIR)

# Evaluate IALS model
ials_recall = evaluate_recommender(ials_model, at=20)
print(f"IALS Model - Recall@20: {ials_recall:.5f}")

IALSRecommender: Loading model from file '/kaggle/working/modelsIALSRecommender'
IALSRecommender: Loading complete
IALS Model - Recall@20: 0.24588


In [8]:
from Recommenders.SLIM.SLIMElasticNetRecommender import SLIMElasticNetRecommender

hybrid_models_dir = os.path.join(paths.MODEL_DIR, 'hybrid')

# Load SLIM model
slim_model = SLIMElasticNetRecommender(URM_train)
slim_model.load_model(os.path.join(hybrid_models_dir, 'slim_URM'))

# Evaluate SLIM model
slim_recall = evaluate_recommender(slim_model, at=20)
print(f"SLIM Model - Recall@20: {slim_recall:.5f}")

SLIMElasticNetRecommender: Loading model from file '/kaggle/working/models/hybrid/slim_URMSLIMElasticNetRecommender'
SLIMElasticNetRecommender: Loading complete
SLIM Model - Recall@20: 0.29005


In [9]:
from Recommenders.KNN.ItemKNNCFRecommender import ItemKNNCFRecommender

# Load SLIM model
knn_model = ItemKNNCFRecommender(URM_train)
knn_model.load_model(os.path.join(paths.MODEL_DIR, 'hybrid'))

# Evaluate KNN model
knn_recall = evaluate_recommender(knn_model, at=20)
print(f"KNN Model - Recall@20: {knn_recall:.5f}")

ItemKNNCFRecommender: Loading model from file '/kaggle/working/models/hybridItemKNNCFRecommender'
ItemKNNCFRecommender: Loading complete
KNN Model - Recall@20: 0.21818


Train SLIM if not saved

In [17]:
from Recommenders.SLIM.SLIMElasticNetRecommender import SLIMElasticNetRecommender

# Load SLIM optuna_study to get best hyperparameters
slim_study = optuna.load_study(study_name=SLIMElasticNetRecommender.RECOMMENDER_NAME+"_refined", storage=paths.OPTUNA_STORAGE)
best_slim_params = slim_study.best_params

# Train SLIM model with best hyperparameters
slim_model = SLIMElasticNetRecommender(URM_train)
slim_model.fit(
    l1_ratio = best_slim_params['l1_ratio'],
    alpha = best_slim_params['alpha'],
    topK = best_slim_params['topK'],
    positive_only = False
)

# Evaluate SLIM model
slim_recall = evaluate_recommender(slim_model, at=20)
print(f"SLIM Model - Recall@20: {slim_recall:.5f}")

SLIMElasticNetRecommender: Processed 3329 (47.8%) in 5.00 min. Items per second: 11.09
SLIMElasticNetRecommender: Processed 6653 (95.5%) in 10.00 min. Items per second: 11.08
SLIMElasticNetRecommender: Processed 6969 (100.0%) in 10.47 min. Items per second: 11.10
SLIM Model - Recall@20: 0.29005


In [19]:
# Save slim model on URM
hybrid_models_dir = os.path.join(paths.MODEL_DIR, 'hybrid')
os.makedirs(hybrid_dir, exist_ok=True)
slim_model.save_model(os.path.join(hybrid_dir, 'slim_URM'))

SLIMElasticNetRecommender: Saving model in file '/kaggle/working/models/hybrid/slim_URMSLIMElasticNetRecommender'
SLIMElasticNetRecommender: Saving complete


Train KNN if not saved

In [24]:
from Recommenders.KNN.ItemKNNCFRecommender import ItemKNNCFRecommender

STUDY_NAME = ItemKNNCFRecommender.RECOMMENDER_NAME

# Must be retrained because the model was not saved
def objective_function(trial):    
    recommender_instance = ItemKNNCFRecommender(URM_train)
    recommender_instance.fit(
        topK=trial.suggest_int("topK", 5, 2000),
        shrink=trial.suggest_int("shrink", 0, 2000),
        similarity="cosine",
        feature_weighting="TF-IDF",
        normalize=True
    )

    return evaluate_recommender(recommender_instance, at=20)

In [25]:
knn_study = hyperparameter_tuning(
    study_name=STUDY_NAME,
    objective_function=objective_function,
    n_trials=100
)

  0%|          | 0/100 [00:00<?, ?it/s]

Similarity column 6969 (100.0%), 1857.40 column/sec. Elapsed time 3.75 sec
[I 2025-11-07 22:54:06,980] Trial 5 finished with value: 0.20078606223219933 and parameters: {'topK': 809, 'shrink': 693}. Best is trial 5 with value: 0.20078606223219933.
Similarity column 6969 (100.0%), 1754.17 column/sec. Elapsed time 3.97 sec
[I 2025-11-07 22:54:57,264] Trial 6 finished with value: 0.19354966098713314 and parameters: {'topK': 1915, 'shrink': 105}. Best is trial 5 with value: 0.20078606223219933.
Similarity column 6969 (100.0%), 1858.31 column/sec. Elapsed time 3.75 sec
[I 2025-11-07 22:55:35,489] Trial 7 finished with value: 0.20356403404362605 and parameters: {'topK': 563, 'shrink': 1582}. Best is trial 7 with value: 0.20356403404362605.
Similarity column 6969 (100.0%), 1789.80 column/sec. Elapsed time 3.89 sec
[I 2025-11-07 22:56:20,435] Trial 8 finished with value: 0.197724499988717 and parameters: {'topK': 1161, 'shrink': 310}. Best is trial 7 with value: 0.20356403404362605.
Similarity 

In [34]:
knn_model = ItemKNNCFRecommender(URM_train)
knn_model.fit(
    topK=79,
    shrink=248,
    similarity="cosine",
    feature_weighting="TF-IDF",
    normalize=True
)

# Evaluate KNN model
knn_recall = evaluate_recommender(knn_model, at=20)
print(f"KNN Model - Recall@20: {knn_recall:.5f}")

Similarity column 6969 (100.0%), 1938.14 column/sec. Elapsed time 3.60 sec
KNN Model - Recall@20: 0.21818


In [35]:
# Save KNN model
knn_model.save_model(os.path.join(paths.MODEL_DIR, 'hybrid'))

ItemKNNCFRecommender: Saving model in file '/kaggle/working/models/hybridItemKNNCFRecommender'
ItemKNNCFRecommender: Saving complete


Train hybrid model

In [29]:
from typing import List
from typing import Dict
from Recommenders.BaseRecommender import BaseRecommender

class HybridRecommender():
    RECOMMENDER_NAME = "HybridRecommender"
    
    def __init__(self, recommenders: Dict[str, BaseRecommender]):
        """
        Parameters
        ----------
        recommenders : dict
            A dictionary mapping model names to recommender instances.
            Example:
                {
                    "ALS": als_model,
                    "ItemKNN": knn_model,
                    "BPR": bpr_model
                }
        """
        self.recommenders = recommenders
        self.set_weights()

    def set_weights(self, weights: Dict[str, float]=None, cutoff_multiplier: float=1.0):      
        if weights is None:
            self.weights = {model_name: 1.0 for model_name in self.recommenders.keys()}
        else:
            self.weights = weights

        self.cutoff_multiplier = cutoff_multiplier

    def recommend(self, user_ids, cutoff=20):
        if type(user_ids) == int:
            user_ids = [user_ids]
        
        cutoff_adjusted = int(cutoff * self.cutoff_multiplier)
        recommendation_list = []

        for user_id in user_ids:
            item_scores = {}
            
            for name, recommender in self.recommenders.items():
                ranking_list, scores_batch = recommender.recommend(
                    user_id, cutoff=cutoff_adjusted, return_scores=True
                )

                # Exctract scores
                scores = scores_batch[0][ranking_list]
        
                # Normalize scores
                max_score = np.max(scores)
                min_score = np.min(scores)
                scores = (scores - min_score) / (max_score - min_score)
                
                # Apply model weight and accumulate
                scores *= self.weights[name]
                for item_id, score in zip(ranking_list, scores):
                    if item_id in item_scores:
                        item_scores[item_id] += score
                    else:
                        item_scores[item_id] = score
    
            # Sort scores and select top-N
            top_items = sorted(item_scores.items(), key=lambda x: x[1], reverse=True)
            top_items = [i[0] for i in top_items]

            recommendation_list.append(top_items[:cutoff])

        return recommendation_list
    
    def save_model(self, folder_path):
        for name, recommender in self.recommenders.items():
            recommender.save_model(folder_path, f"{name}_submodel")

    def load_model(self, recommender_classes, folder_path):
        self.recommenders = {}
        for name, recommender_class in recommender_classes.items():
            self.recommenders['name'] = recommender_class.load_model(folder_path, f"{name}_submodel")

In [30]:
from optuna.exceptions import TrialPruned

STUDY_NAME = HybridRecommender.RECOMMENDER_NAME+"_8"

# Must be retrained because the model was not saved
def hybrid_objective(trial):
    # Create hybrid model
    hybrid_model = HybridRecommender(
        recommenders={
            "IALS" : ials_model,
            "SLIM" : slim_model, 
            "KNN"  : knn_model
        }
    )

    # Let optuna sample hyperparameters
    cutoff_multiplier = trial.suggest_float("cutoff_multiplier", 1, 5.0)
    w_knn  = trial.suggest_float("knn_weight",  0.0, 0.4)
    w_ials = trial.suggest_float("ials_weight", 0.0, 0.4)
    w_slim = 1.0 - w_knn - w_ials

    # Set hyperparameter
    hybrid_model.set_weights(
        weights={
            "IALS" : w_ials,
            "SLIM" : w_slim, 
            "KNN"  : w_knn
        },
        cutoff_multiplier=cutoff_multiplier
    )

    # Evaluate hyperparameters
    return evaluate_recommender(hybrid_model, at=20)

In [31]:
# Perform hyperparameter tuning
save_results, optuna_study = hyperparameter_tuning(
    hybrid_objective,
    study_name=STUDY_NAME,
    n_trials=200
)

  0%|          | 0/200 [00:00<?, ?it/s]

[I 2025-11-08 01:51:16,817] Trial 0 finished with value: 0.26345227890428274 and parameters: {'cutoff_multiplier': 3.9078179982018595, 'knn_weight': 0.3923323807917937, 'ials_weight': 0.3883201079049591}. Best is trial 0 with value: 0.26345227890428274.
[I 2025-11-08 01:52:13,192] Trial 1 finished with value: 0.2917282797037487 and parameters: {'cutoff_multiplier': 3.471658820657245, 'knn_weight': 0.0803758248742894, 'ials_weight': 0.10384443172712846}. Best is trial 1 with value: 0.2917282797037487.
[I 2025-11-08 01:53:10,335] Trial 2 finished with value: 0.2879249932508243 and parameters: {'cutoff_multiplier': 3.5894132659145357, 'knn_weight': 0.1195130220043076, 'ials_weight': 0.3293941630669005}. Best is trial 1 with value: 0.2917282797037487.
[I 2025-11-08 01:54:06,642] Trial 3 finished with value: 0.29074176434181714 and parameters: {'cutoff_multiplier': 4.296069853663582, 'knn_weight': 0.08956467790580223, 'ials_weight': 0.2576901078540023}. Best is trial 1 with value: 0.2917282

In [14]:
optuna_study = optuna.load_study(study_name=HybridRecommender.RECOMMENDER_NAME+"_8", storage=paths.OPTUNA_STORAGE)
best_hybrid_params = optuna_study.best_params
best_hybrid_params

{'cutoff_multiplier': 3.339605055645294,
 'knn_weight': 0.023923505254710455,
 'ials_weight': 0.18648419882498404}

In [15]:
optuna.visualization.plot_optimization_history(optuna_study)

In [33]:
optuna.visualization.plot_param_importances(optuna_study)

In [34]:
optuna.visualization.plot_parallel_coordinate(optuna_study)

In [30]:
# Initialize the best model
best_param = optuna_study.best_trial.params

hybrid_model = HybridRecommender(
    recommenders={
        "IALS" : ials_model,
        "SLIM" : slim_model, 
        "KNN"  : knn_model
    }
)

cutoff_multiplier = best_param["cutoff_multiplier"]
w_knn  =best_param["knn_weight"]
w_ials = best_param["ials_weight"]
w_slim = 1.0 - w_knn - w_ials

# Set hyperparameter
hybrid_model.set_weights(
    weights={
        "IALS" : w_ials,
        "SLIM" : w_slim, 
        "KNN"  : w_knn
    },
    cutoff_multiplier=cutoff_multiplier
)

In [31]:
hybrid_recall = evaluate_recommender(hybrid_model, at=20)
print(f"Hybrid Model - Recall@20: {hybrid_recall:.5f}")

Hybrid Model - Recall@20: 0.29293


In [32]:
import pandas as pd

# Generate recommendations for the test set
user_ids_test = pd.read_csv(paths.CHALLENGE_USER_IDS_TEST)
ids = user_ids_test["user_id"].values

recommendations = hybrid_model.recommend(ids, cutoff=20)

os.makedirs(paths.SUBMISSIONS, exist_ok=True)
with open(os.path.join(paths.SUBMISSIONS, HybridRecommender.RECOMMENDER_NAME+"_no_val" + ".csv"), "w") as f:
    f.write("user_id,item_list\n")
    for user_id, rec_list in zip(ids, recommendations):
        f.write(f"{user_id},{' '.join([str(item) for item in rec_list])}\n")

In [44]:
# Load the best SLIM model, IALS model is trained looking at validation already
slim_best = SLIMElasticNetRecommender(URM_train+URM_validation)
slim_best.load_model(paths.MODEL_DIR)

# Train ItemKNNCF model
knn_study = optuna.load_study(study_name=ItemKNNCFRecommender.RECOMMENDER_NAME, storage=paths.OPTUNA_STORAGE)

knn_best = ItemKNNCFRecommender(URM_train+URM_validation)
knn_best.fit(
    topK=knn_study.best_trial.params["topK"],
    shrink=knn_study.best_trial.params["shrink"],
    similarity="cosine",
    feature_weighting="TF-IDF",
    normalize=True
)

SLIMElasticNetRecommender: Loading model from file '/kaggle/working/modelsSLIMElasticNetRecommender'
SLIMElasticNetRecommender: Loading complete
Similarity column 6969 (100.0%), 1399.73 column/sec. Elapsed time 4.98 sec


In [45]:
slim_best, knn_best

(<Recommenders.SLIM.SLIMElasticNetRecommender.SLIMElasticNetRecommender at 0x79d783949f10>,
 <Recommenders.KNN.ItemKNNCFRecommender.ItemKNNCFRecommender at 0x79d845eee410>)

In [46]:
# Use SLIM and KNN retrained on validation
hybrid_model_val = HybridRecommender(
    recommenders={
        "IALS" : ials_model,
        "SLIM" : slim_best, 
        "KNN"  : knn_best
    }
)

# Set hyperparameter
hybrid_model_val.set_weights(
    weights={
        "IALS" : w_ials,
        "SLIM" : w_slim, 
        "KNN"  : w_knn
    },
    cutoff_multiplier=cutoff_multiplier
)

In [48]:
import pandas as pd

# Generate recommendations for the test set
user_ids_test = pd.read_csv(paths.CHALLENGE_USER_IDS_TEST)
ids = user_ids_test["user_id"].values

recommendations = hybrid_model_val.recommend(ids, cutoff=20)

os.makedirs(paths.SUBMISSIONS, exist_ok=True)
with open(os.path.join(paths.SUBMISSIONS, HybridRecommender.RECOMMENDER_NAME+"_val" + ".csv"), "w") as f:
    f.write("user_id,item_list\n")
    for user_id, rec_list in zip(ids, recommendations):
        f.write(f"{user_id},{' '.join([str(item) for item in rec_list])}\n")